In [1]:
try:
    import pylibjpeg
except:
    !mkdir -p /root/.cache/torch/hub/checkpoints/
    !cp ../input/effdet-models/efficientnet_v2s_ra2_288-a6477665.pth /root/.cache/torch/hub/checkpoints/
    !pip install ../input/effdet-models/pydicom-2.3.0-py3-none-any.whl
    !pip install ../input/effdet-models/pylibjpeg-1.4.0-py3-none-any.whl
    !pip install ../input/effdet-models/pylibjpeg_libjpeg-1.3.2-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
    !pip install ../input/effdet-models/python_gdcm-3.0.19-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
    !pip install ../input/effdet-models/pycocotools-2.0.2-cp37-cp37m-linux_x86_64.whl
#     !pip install ../input/rsna-2022-whl/torch-1.12.1-cp37-cp37m-manylinux1_x86_64.whl
#     !pip install ../input/rsna-2022-whl/torchvision-0.13.1-cp37-cp37m-manylinux1_x86_64.whl

cp: cannot stat '../input/effdet-models/efficientnet_v2s_ra2_288-a6477665.pth': No such file or directory
Processing /kaggle/input/effdet-models/pydicom-2.3.0-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/effdet-models/pydicom-2.3.0-py3-none-any.whl'

Processing /kaggle/input/effdet-models/pylibjpeg-1.4.0-py3-none-any.whl
ERROR: Could not install packages due to an OSError: [Errno 2] No such file or directory: '/kaggle/input/effdet-models/pylibjpeg-1.4.0-py3-none-any.whl'

ERROR: pylibjpeg_libjpeg-1.3.2-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl is not a supported wheel on this platform.
ERROR: python_gdcm-3.0.19-cp37-cp37m-manylinux_2_17_x86_64.manylinux2014_x86_64.whl is not a supported wheel on this platform.
ERROR: pycocotools-2.0.2-cp37-cp37m-linux_x86_64.whl is not a supported wheel on this platform.


In [2]:
import pandas as pd
import torch
# import matplotlib.pyplot as plt
# import seaborn as sns
# import ast
import re
import sys

import pydicom as dicom
import pylibjpeg

effdet_path = "../input/effdet-models/effdet"
sys.path.append(effdet_path)


timm_path = "../input/effdet-models/timm-pytorch-image-models"
sys.path.append(timm_path)
import timm
from timm.data import IMAGENET_DEFAULT_MEAN, IMAGENET_DEFAULT_STD
import matplotlib.pyplot as plt
# from matplotlib import patches
# import sklearn
import os
from tqdm import tqdm

omega_path = "../input/effdet-models/omegaconf"
sys.path.append(omega_path)
from omegaconf import OmegaConf

effunet_path = "../input/effdet-models/efficientunet-pytorch-0.0.6"
sys.path.append(effunet_path)

yolo_path = "../input/effdet-models/yolov7"
sys.path.append(yolo_path)

# from effdet import create_model

import glob
# import sklearn
import math
import random
import numpy as np

from PIL import Image

# import cv2
# import albumentations as A
# from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch import optim
from torchvision import transforms
# import torchvision.models as models
import torchvision.transforms as T
import torchvision.transforms.functional as TF
# from transformers import get_cosine_schedule_with_warmup

import warnings

warnings.filterwarnings('ignore')
# from sklearn import metrics, model_selection, preprocessing

# from sklearn.model_selection import GroupKFold


# pos_weight = torch.tensor(pos_weight)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
device

ModuleNotFoundError: No module named 'pylibjpeg'

In [3]:


IMAGES_DIR='../input/rsna-2022-cervical-spine-fracture-detection/test_images'
TRAIN_IMAGES_PATH='../input/rsna-2022-cervical-spine-fracture-detection/train_images'
TEST_IMAGES_PATH='../input/rsna-2022-cervical-spine-fracture-detection/test_images'

In [4]:

segmentation_checkpoint="../input/effdet-models/axial_segmentation_effseg_132508-epoch-100.pth"
axial_det_checkpoint1="../input/effdet-models/axial_detection_effdet_134352-epoch-52.pth"
axial_det_checkpoint2="../input/effdet-models/axial_detection_effdet_001015-epoch-150.pth"
axial_yolo_checkpoint1="../input/effdet-models/yolo_custom4_epoch_099.pt"

In [5]:
from models.experimental import attempt_load
yolo_model = attempt_load(axial_yolo_checkpoint1, map_location=device)
yolo_model = yolo_model.eval()

ModuleNotFoundError: No module named 'models'

In [6]:
# # for test, use these
# train_df = pd.read_csv('../input/rsna-2022-cervical-spine-fracture-detection/train.csv').set_index('StudyInstanceUID')
# train_df.head()

# IMAGES_DIR='../input/rsna-2022-cervical-spine-fracture-detection/train_images'
# TRAIN_IMAGES_PATH='../input/rsna-2022-cervical-spine-fracture-detection/train_images'
# TEST_IMAGES_PATH='../input/rsna-2022-cervical-spine-fracture-detection/test_images'

# test_slices = glob.glob(f'{TRAIN_IMAGES_PATH}/1.2.826.0.1.3680043.21*/*')
# test_slices = [re.findall(f'{TRAIN_IMAGES_PATH}/(.*)/(.*).dcm', s)[0] for s in test_slices]
# df_test_slices = pd.DataFrame(data=test_slices, columns=['StudyInstanceUID', 'Slice']).astype({'Slice': int})

# df_test_slices

# #test end

test_slices = glob.glob(f'{TEST_IMAGES_PATH}/*/*')
test_slices = [re.findall(f'{TEST_IMAGES_PATH}/(.*)/(.*).dcm', s)[0] for s in test_slices]
df_test_slices = pd.DataFrame(data=test_slices, columns=['StudyInstanceUID', 'Slice']).astype({'Slice': int})
df_test_slices

NameError: name 'glob' is not defined

In [7]:
df_test_slices = df_test_slices.set_index('StudyInstanceUID')
df_test_slices["Start"] = df_test_slices.groupby('StudyInstanceUID').apply(lambda df: df.Slice.min())
# df_test_slices["End"] = df_test_slices.groupby('StudyInstanceUID').apply(lambda df: df.Slice.max())
df_test_slices = df_test_slices.sort_values(['StudyInstanceUID', 'Slice']).reset_index(drop=False)
df_test_slices.head()

NameError: name 'df_test_slices' is not defined

In [8]:
def rescale_img_to_hu(dcm_ds):
    """Rescales the image to Hounsfield unit.
    """
    return dcm_ds.pixel_array * dcm_ds.RescaleSlope + dcm_ds.RescaleIntercept

def normalize_hu(data):
    # normalize to 0-1
    # return (data - data.min()) / data.max()
    data = np.clip(data, a_min=-2242, a_max=2242) / 4484 + 0.5
    return data

def normalize_hu_t(data):
    # normalize to 0-1
    # return (data - data.min()) / data.max()
    return np.clip(data, a_min=-2242., a_max=2242.) / 2242.

def normalize_hu_tensor(data):
    # normalize to -1 to 1
    # return (data - data.min()) / data.max()
    return torch.clip(data, min=-2242., max=2242.) / 2242.
#     return data

def load_dicom(path):
    """
    This supports loading both regular and compressed JPEG images. 
    See the first sell with `pip install` commands for the necessary dependencies
    """
    ds=dicom.dcmread(path) 

    img = rescale_img_to_hu(ds)

    return img, ds.PixelSpacing[0]


In [9]:
class DcmDataSet(torch.utils.data.Dataset):    
    def __init__(self, df, path, image_size=512):
        super().__init__()
        self.df = df
        self.path = path
        
        self.len = len(self.df)
        self.image_size = image_size
        
        # 이걸 무조건 해야 덴다.
        # 어떤 이미지들이 512 가 아니다. 그래서 여기서 일단 바까줘야 한다. 
        self.transform = T.Resize((image_size, image_size))
        
    def __getitem__(self, i):
        
        try:
            s = self.df.iloc[i] # 지금꺼
            gpath = os.path.join(self.path, s.StudyInstanceUID, f'{s.Slice}.dcm')
            
            g, pixel_spacing = load_dicom(gpath) 
            g = normalize_hu_t(g)
            

        except Exception as ex:
            print(ex)
            return None, None, None
        
        return self.transform(torch.as_tensor(g, dtype=torch.float).unsqueeze(0)), pixel_spacing, s.Slice == s.Start
    def __len__(self):
        return self.len

ds = DcmDataSet(df_test_slices, IMAGES_DIR)


NameError: name 'df_test_slices' is not defined

In [10]:
batch_size = 16
dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=min(os.cpu_count(), batch_size))

x, pixel_spacings, is_start = next(iter(dl))
print(x.min(), x.max())
print(x.shape)
print(pixel_spacings, is_start)

NameError: name 'DataLoader' is not defined

In [11]:
from efficientunet import *
def get_axial_segmentation_model(checkpoint):
    model = get_efficientunet_b5(out_channels=2, concat_input=True, pretrained=False)
    
    state = torch.load(checkpoint, map_location=torch.device(device))
    model.load_state_dict(state["model"])
    model.eval()
    return model.to(device)
    
seg_model = get_axial_segmentation_model(segmentation_checkpoint)

ModuleNotFoundError: No module named 'efficientunet'

In [12]:
from effdet import create_model

def get_axial_detection_model(checkpoint, image_size=512):
    model = create_model('efficientdetv2_ds' , bench_task='predict' , num_classes=1 , image_size=(image_size, image_size), pretrained=False, max_det_per_image=1)

    state = torch.load(checkpoint, map_location=torch.device(device))
    model.load_state_dict(state["model"])
    
    model = model.eval()
    return model.to(device)

def get_axial_detection_models(checkpoints):
    models = []
    for checkpoint in checkpoints:
        model = get_axial_detection_model(checkpoint)
        models.append(model)
    return models

effdet_models = get_axial_detection_models([axial_det_checkpoint1])
# effdet_models = get_axial_detection_models([axial_det_checkpoint1, axial_det_checkpoint2])

ModuleNotFoundError: No module named 'effdet'

In [13]:
IMAGE_SIZES=[640, 512]
det_model_names=['yolo', 'effdet'] # effdet
det_models = [yolo_model] + effdet_models
# det_models = [yolo_model]

NameError: name 'yolo_model' is not defined

In [14]:
def get_axial_boundary_from_segmentation(seg, pixel_spacing, throw=100, tol=0.2, max_mm=100):
    """
    seg : H x W
    """
    image_size = seg.shape[0]
    min_size = min(image_size, max_mm / pixel_spacing)
    
    rows, columns = seg.nonzero(as_tuple=True)
    rows.sort()
    columns.sort()
    
    throw = min(len(rows) // 2, throw)
    
    if(len(rows)) == 0:
        return torch.tensor([0, 0, image_size, image_size]).to(device)
    
    xmin, xmax = columns[throw], columns[-throw]
    ymin, ymax = rows[throw], rows[-throw]
    
    w = (xmax - xmin) * (1 + tol)
    h = (ymax - ymax) * (1 + tol)
    new_size = max(w, h, min_size)
    new_size = min(image_size, new_size)
    
    xcenter, ycenter = (xmax + xmin) / 2, (ymax + ymin) / 2
    
    xmin = torch.min(torch.tensor(image_size - new_size), xcenter - new_size / 2)
    xmin = xmin.clip(min=0)
    
    ymin = torch.min(torch.tensor(image_size - new_size), ycenter - new_size / 2)
    ymin = ymin.clip(min=0)
    
    return torch.stack([xmin, ymin, xmin + new_size, ymin + new_size])

In [15]:
def predict_seg(x, model, seg_img_size=256):
    """
    return: N x 1 x H x W
    """
    x = TF.resize(x, (seg_img_size, seg_img_size))
    logits = model(x)

    classification_score, mse_score = logits.sigmoid().chunk(2, dim=1)
    classification_pred = classification_score.gt(0.5).float()
    pred = (classification_pred * mse_score)
    
    return pred

In [16]:
def get_axial_boundary_from_seg(segs, pixel_spacings, seg_img_size=256):
    boundary_list = []
    for i in range(segs.shape[0]):
        seg = segs[i, 0, :, :]
        
        boundary = get_axial_boundary_from_segmentation(seg, pixel_spacings[i], throw=int(100. / 512. * seg_img_size), tol=0.2, max_mm=100. / 512. * seg_img_size)
        boundary_list.append(boundary)
    boundary_list = torch.stack(boundary_list, axis=0) * (512. / seg_img_size)
    return boundary_list

In [17]:

def convert_yolo_result(pred):
#     pred = model(x)[0]
    max_indices = torch.argmax(pred[:, :, 4], dim=1)
    max_values = pred[torch.arange(pred.shape[0]), max_indices, :] # N x 6
    
    bboxes, scores = max_values[:, :4], max_values[:, 4]
    bboxes[:, 2] += bboxes[:, 0]
    bboxes[:, 3] += bboxes[:, 1]
    return bboxes, scores

def predict_det(x, model):
    
    pred_result = model(x) # N x 1 x 6
    
    if isinstance(pred_result, tuple) is True:
        # yolo
        return convert_yolo_result(pred_result[0])
    else:
        return pred_result[:, 0, :4], pred_result[:, 0, 4]
        
    
    
def crop_resize_images(imgs_tensor, boundary_list, img_size=512):
    croped_list = []
    for i in range(imgs_tensor.shape[0]):
        xmin, ymin, xmax, ymax = boundary_list[i, :]
        xmin, ymin, xmax, ymax = int(xmin), int(ymin), int(xmax), int(ymax)
        # print(xmin, ymin, xmax, ymax)
        croped = TF.crop(imgs_tensor[i, :, :, :], top=ymin, left=xmin, height=ymax-ymin, width=xmax-xmin)
        croped = TF.resize(croped, (img_size, img_size))
        croped_list.append(croped)
        
    return torch.stack(croped_list, 0)

def get_original_bbox(bbox, boundary, image_size=512.):
    scale = image_size / (boundary[:, [2]] - boundary[:, [0]])
    
    org_bbox = bbox / scale
    org_bbox[:, 0] += boundary[:, 0]
    org_bbox[:, 1] += boundary[:, 1]
    org_bbox[:, 2] += boundary[:, 0]
    org_bbox[:, 3] += boundary[:, 1]
    
    return org_bbox



In [18]:
def get_bbox_class(seg, bbox):
    """
    label 은 0.125 의 단위로, 
    seg: H x W
    bbox: [xmin, ymin, xmax, ymax]
    """
    xmin, ymin, xmax, ymax = bbox.int()
    area = seg[ymin:ymax, xmin:xmax]
    
    # print(area)
    result = torch.mean(area[area>0])
    result = torch.round(result / 0.125)
    
    return result

In [19]:
def get_bbox_class_list(seg_list, seg_bboxes):
    class_list = []
    for i in range(seg_list.shape[0]):
        class_index = get_bbox_class(seg_list[i, :, :], seg_bboxes[i, :])
        class_list.append(class_index)
        
    return torch.stack(class_list)

In [20]:

def get_class_score(scores, class_list, eps=1e-2):
    result = scores.new_zeros((scores.shape[0], 8)) + eps
    class_list = torch.nan_to_num(class_list).long()
    result[torch.arange(scores.shape[0]), class_list] = scores
    
    return result

def check_detection_result(det_result, img_size=512., threshold=0.2):
    # throw big bboxes
    areas = (det_result[:, 2] - det_result[:, 0]) * (det_result[:, 3] - det_result[:, 1]) / (img_size * img_size)
    # print(areas)
    big_indices = torch.argwhere(areas > threshold)
    det_result[big_indices, 4] = 0.
    return det_result


In [21]:
def cal_loss(prob, label):
    
    pos_weight = np.array([14, 2, 2, 2, 2, 2, 2, 2])
    neg_weight = np.array([7, 1, 1, 1, 1, 1, 1, 1])
    
    score = pos_weight * label * np.log(prob) + neg_weight * (1 - label) * np.log(1 - prob)
    
    weight_total = pos_weight * label + neg_weight * (1 - label)
    
    return -score.sum(axis=1) / weight_total.sum(axis=1)

In [22]:
def predict():
    
    with torch.no_grad():
        predictions = []
        
#         tqdm_iter = tqdm(dl)

        x0, _, _ = ds[0]
        x1, _, _ = ds[1]
        x0, x1 = x0.to(device), x1.to(device)
        prev2 = torch.stack((x0, x1))
#         print(prev2.shape)

#         return
        for x, pixel_spacings, is_starts in tqdm(dl):
            
            # x : N x 1 x 512 x 512
            x = x.to(device)
            
            # (N+2), 1, 512, 512
            x = torch.cat((prev2, x), dim=0)
            
            r = x[:-2, :, :, :]
            g = x[1:-1, :, :, :]
            b = x[2:, :, :, :]
            
            start_indices = torch.argwhere(is_starts)
            r[start_indices, :, :, :] = b[start_indices, :, :, :]
            g[start_indices, :, :, :] = b[start_indices, :, :, :]

            prev2 = b[-2:, :, :, :]
            
            x = torch.cat((r, g, b), dim=1)
            
            batch_probs = x.new_zeros((x.shape[0], 8)) + 1e-2
            
            seg_result = predict_seg(x, seg_model)  # N x 1 x 256 x 256
            
            active_indices = seg_result.sum(axis=[1, 2, 3]).nonzero().reshape(-1)
            if active_indices.numel() == 0:
                predictions.append(batch_probs)
                continue
            
#             tqdm_iter.set_description(f'{active_indices.numel()}')
            if active_indices.numel() != batch_size:
                x = x[active_indices, :, :, :]
                seg_result = seg_result[active_indices, :, :, :]
                pixel_spacings = pixel_spacings[active_indices]
            
            
            axial_boundary = get_axial_boundary_from_seg(seg_result, pixel_spacings, seg_img_size=256)  # N x 4
            
            for i, det_model in enumerate(det_models):
                
                croped_x = crop_resize_images(x, axial_boundary, IMAGE_SIZES[i])
                
                det_model_name = det_model_names[i]
                if det_model_name == 'yolo':
                    croped_x = croped_x * 0.5 + 0.5
                bboxes, scores = predict_det(croped_x, det_model)
                
                # unit 512 x 512
                bboxes = get_original_bbox(bboxes, axial_boundary, IMAGE_SIZES[i]) 
                class_list = get_bbox_class_list(seg_result[:, 0, :, :], bboxes / 2)
                probs = get_class_score(scores, class_list) # N x 7
                
                batch_probs[active_indices, :] += probs

            predictions.append(batch_probs / len(det_models))
        
        return torch.concat(predictions).cpu().numpy()
predictions = predict()
# predictions

NameError: name 'ds' is not defined

In [23]:
df_effnet_pred = pd.DataFrame(
    data=predictions, columns=['patient_overall'] + [f'C{i}' for i in range(1, 8)]
)
df_test_pred = pd.concat([df_test_slices, df_effnet_pred], axis=1).sort_values(['StudyInstanceUID', 'Slice'])
df_patient_pred = df_test_pred.groupby('StudyInstanceUID').apply(lambda df: df.max())

# clip
clip_value = 1e-3
df_patient_pred[[f'C{i}' for i in range(1, 8)]] = df_patient_pred[[f'C{i}' for i in range(1, 8)]].clip(lower=clip_value, upper=1-clip_value)

# overall 를 max 로 정하기
df_patient_pred["patient_overall"] = df_patient_pred[[f'C{i}' for i in range(1, 8)]].max(axis=1)

#overall 를 1-neg 로 정하기
# df_patient_pred["patient_overall"] = np.clip(1 - (1 - df_patient_pred[[f'C{i}' for i in range(1, 8)]]).product(axis=1), a_min=clip_value, a_max=1-clip_value)

df_patient_pred = df_patient_pred[['patient_overall'] + [f'C{i}' for i in range(1, 8)]]
df_patient_pred

NameError: name 'predictions' is not defined

In [24]:
# prob = df_patient_pred.values
# label = train_df.loc[df_patient_pred.index].values

# losses = cal_loss(prob, label)
# print(np.mean(losses))
# list(losses)


In [25]:
# df = pd.read_csv('../input/rsna-2022-cervical-spine-fracture-detection/test.csv')
df_test = pd.read_csv(f'../input/rsna-2022-cervical-spine-fracture-detection/test.csv')

if df_test.iloc[0].row_id == '1.2.826.0.1.3680043.10197_C1':
    # test_images and test.csv are inconsistent in the dev dataset, fixing labels for the dev run.
    df_test = pd.DataFrame({
        "row_id": ['1.2.826.0.1.3680043.22327_C1', '1.2.826.0.1.3680043.25399_C1', '1.2.826.0.1.3680043.5876_patient_overall'],
        "StudyInstanceUID": ['1.2.826.0.1.3680043.22327', '1.2.826.0.1.3680043.25399', '1.2.826.0.1.3680043.5876'],
        "prediction_type": ["C1", "C1", "patient_overall"]}
    )

df_test

,StudyInstanceUID,prediction_type,row_id
0,1.2.826.0.1.3680043.6200,patient_overall,1.2.826.0.1.3680043.6200_patient_overall
1,1.2.826.0.1.3680043.27262,patient_overall,1.2.826.0.1.3680043.27262_patient_overall
2,1.2.826.0.1.3680043.12351,patient_overall,1.2.826.0.1.3680043.12351_patient_overall
3,1.2.826.0.1.3680043.1363,patient_overall,1.2.826.0.1.3680043.1363_patient_overall
4,1.2.826.0.1.3680043.4859,patient_overall,1.2.826.0.1.3680043.4859_patient_overall
...,...,...,...
14531,1.2.826.0.1.3680043.21684,C7,1.2.826.0.1.3680043.21684_C7
14532,1.2.826.0.1.3680043.4786,C7,1.2.826.0.1.3680043.4786_C7
14533,1.2.826.0.1.3680043.14341,C7,1.2.826.0.1.3680043.14341_C7
14534,1.2.826.0.1.3680043.12053,C7,1.2.826.0.1.3680043.12053_C7


In [26]:
df_sub = df_test.copy()
df_sub = df_sub.set_index('StudyInstanceUID').join(df_patient_pred)
df_sub['fractured'] = df_sub.apply(lambda r: r[r.prediction_type], axis=1)
df_sub

NameError: name 'df_patient_pred' is not defined

In [27]:
df_sub[['row_id', 'fractured']].to_csv('submission.csv', index=False)

KeyError: "['fractured'] not in index"